# Fase 5 — Conclusiones y Recomendaciones de Negocio

**Proyecto:** Desarrollo de un Proyecto de Análisis de Datos y Modelo Predictivo para una Aplicación de Negocio
**Materia:** Análisis de Datos / Inteligencia de Negocios / Aprendizaje Computacional · ITM
**Fecha:** Mayo 2026
**Dataset:** Olist Brazilian E-Commerce (2017-2018, ~98 666 órdenes)

---

## Contenido

1. Síntesis ejecutiva
2. Hallazgos transversales (fases 1 → 4)
3. Recomendaciones de negocio mapeadas a las preguntas PN1-PN10
4. Hoja de ruta de implementación (90 / 180 / 365 días)
5. Limitaciones del trabajo
6. Consideraciones éticas
7. Próximos pasos sugeridos
8. Cierre


## 1. Síntesis ejecutiva

Olist es un marketplace **operacionalmente saludable** pero con **dos
brechas accionables** muy definidas:

- **Logística diferencial entre regiones**: SP/PR/MG entregan en ~9-12
  días; estados del norte (RR, AM, AP) en ~25-30 días. La distancia
  geográfica al sudeste se traduce directamente en satisfacción y
  retención.
- **Concentración de la demanda en 3 estados**: SP+RJ+MG concentran el
  66.6% de las órdenes. Cualquier evento adverso en esa región afecta
  desproporcionadamente al marketplace completo.

El **modelo predictivo** desarrollado en la Fase 4 alcanza
**ROC-AUC = 0.705** y **F1 = 0.837** sobre el test, permitiendo
*anticipar* qué órdenes activas terminarán con reseña negativa con
suficiente margen para intervenir antes de que el cliente publique
la crítica.

Las recomendaciones que siguen están **priorizadas por impacto
esperado en CSAT** (la métrica que un *retail head* monitorea cada
mañana) y respaldadas, cada una, con la evidencia técnica de las
fases anteriores.


## 2. Hallazgos transversales

### 2.1 Fase 1 — Recopilación y ETL

- Se evaluaron **5 datasets candidatos** con criterios ponderados.
  Olist ganó por su modelo multi-tabla y riqueza de problemas de
  negocio (4.55/5 en la matriz).
- El **modelo dimensional** (esquema estrella con 6 dimensiones y 2
  tablas de hechos) sirve simultáneamente al análisis (Pandas), a la
  BI (Power BI) y al modelo predictivo (sklearn). Una sola fuente de
  verdad.
- **12/12 validaciones** de integridad pasaron post-ETL: claves
  primarias únicas, foreign keys 100% válidas, sin valores negativos
  en variables monetarias.

### 2.2 Fase 2 — EDA

- **CSAT = 77.6%** (2.4 p.p. debajo de la meta del 80%). El 22.4%
  insatisfecho deja un espacio muy claro de gestión.
- **OTIF = 91.9%** (3.1 p.p. debajo del 95%). La logística es el
  driver dominante.
- Las **5 hipótesis estadísticas** se rechazaron (p < 0.05),
  confirmando estadísticamente que:
  - H1: el tiempo de entrega medio difiere entre reseñas positivas y negativas.
  - H2: `is_late` y la reseña están asociadas.
  - H3: el método de pago se asocia con la satisfacción.
  - H4: existe correlación entre cuotas y review_score.
  - H5: el tiempo medio de entrega difiere entre el sudeste y el resto del país.
- **Plan de outliers documentado** (sección 12.3 de la Fase 2) y
  aplicado idénticamente en la Fase 4. Cero ambigüedad.

### 2.3 Fase 3 — BI

- **Dashboard de 3 páginas** (Resumen, Operación, Satisfacción) en
  Power BI más su *espejo HTML*. Cada visualización está mapeada a
  una pregunta de negocio.
- **30+ medidas DAX** documentadas y validadas contra los cálculos en
  Python (diferencias < 0.05 p.p.).
- Capa semántica diseñada para soportar drill-through hasta el detalle
  de una orden individual.

### 2.4 Fase 4 — Modelado

- **XGBoost** elegido como modelo final: ROC-AUC = 0.705, F1 = 0.837.
- **Permutation importance** confirma que las 3 features más
  predictivas son **logísticas**: `delivery_days_real`, `n_items`,
  `delivery_delay_days`.
- El modelo aporta una **señal accionable** para intervención
  preventiva: identifica el ~22% de órdenes negativas con una
  precisión 40% superior al baseline trivial.


## 3. Recomendaciones de negocio

Cada recomendación está **anclada a una pregunta de negocio (PN)**,
respaldada con evidencia técnica y acompañada de una **métrica de
seguimiento** para validar su efecto.

### R1 — Abrir hubs logísticos regionales en el Nordeste (PN3)
**Evidencia.** SP entrega en 8.7 días; estados del nordeste (MA, AL,
SE, PA, RR) lo hacen en 21-29 días. La hipótesis H5 confirma la
diferencia (p < 0.001). El modelo predictivo asigna alta importancia
a `delivery_days_real`.

**Acción.** Pilotar un **hub logístico en Recife o Salvador** con
acuerdo de cross-docking con un *carrier* local. Objetivo: reducir
de 22 a 14 días el tiempo medio de entrega en el nordeste.

**KPI de seguimiento.** Tiempo medio de entrega NE → ≤ 14 d en 6 meses.
**Impacto CSAT esperado.** +1.5 a +2.0 p.p. en el CSAT global.

---

### R2 — Auditar categorías con CSAT < 75% (PN4)
**Evidencia.** Categorías como `office_furniture` (63%),
`bed_bath_table` (74%) y `telephony` (75%) tienen CSAT muy por debajo
de la media. El heatmap categoría × estado muestra que el problema
es transversal a la geografía.

**Acción.** Crear un **programa de auditoría de vendedores** en estas
3 categorías: revisión de fotos, descripciones, tiempos de
disponibilidad y atención post-venta. Suspensión temporal de los
vendedores con CSAT < 60% durante 3 meses consecutivos.

**KPI.** CSAT por categoría auditada → +5 p.p. en 6 meses.
**Impacto CSAT esperado.** +0.8 p.p. global (con las 3 categorías
representando ~17k órdenes/año).

---

### R3 — Sistema de alertas tempranas para órdenes en riesgo (PN10)
**Evidencia.** El modelo XGBoost (Fase 4) identifica las órdenes con
alta probabilidad de reseña negativa antes de que el cliente la deje.
El top-50 *órdenes en riesgo* del test set tiene una concentración
del 78% de reseñas reales negativas.

**Acción.** Integrar el modelo al CRM:
1. Score diario sobre las órdenes con `order_status` ∈
   {processing, shipped} y `delivery_days_estimated` próxima.
2. Las órdenes con `P(negativa) > 0.6` entran a una cola de
   atención preventiva (notificación al cliente, seguimiento
   logístico, oferta de cupón si aplica).
3. Re-entrenar el modelo trimestralmente con datos nuevos para
   evitar *concept drift*.

**KPI.** % de órdenes "rescatadas" (P alto → reseña positiva final).
**Impacto CSAT esperado.** +1.0 a +1.5 p.p. — depende del % de
órdenes intervenidas que cambian de signo.

---

### R4 — Diversificación geográfica (PN8)
**Evidencia.** Top-3 estados concentran 66.6% de las órdenes. Riesgo
de dependencia.

**Acción.** Campaña de captación de vendedores en estados del centro
y sur (RS, SC, MS, GO, DF) con incentivos de comisión reducida los
primeros 3 meses.

**KPI.** % órdenes top-3 → ≤ 60% en 12 meses (índice Herfindahl).
**Impacto.** Reducción del riesgo operativo; mejora secundaria del
tiempo de entrega en esas regiones por proximidad al vendedor.

---

### R5 — Optimizar la experiencia de cuotas/pagos (PN5 + EDA 11 H3, H4)
**Evidencia.** La hipótesis H3 (Fase 2 11) confirma que el método
de pago se asocia con la satisfacción (chi-cuadrado, p < 0.05) — esto
es lo que el dashboard página 3 muestra como "CSAT por método de
pago". La hipótesis H4 va un paso más allá y confirma la correlación
de Spearman entre número de cuotas y `review_score`: los clientes que
financian a más cuotas son ligeramente más exigentes.

**Acción.** Implementar comunicación proactiva al cliente que paga a
>6 cuotas: confirmaciones periódicas, recordatorios del valor, oferta
de soporte premium. Estos clientes representan el 18% del volumen
pero el 24% de las reseñas negativas.

**KPI.** CSAT entre clientes con cuotas > 6 → del 73% al 78%.
**Impacto CSAT esperado.** +0.4 p.p. global.

---

### R6 — Comunicación preventiva en Black Friday (PN6 + EDA 7)
**Evidencia.** El "Volumen mensual de órdenes" del dashboard
(página 1) muestra el pico de noviembre 2017 (~3× del nivel base);
el análisis temporal del EDA (7) lo identifica como el Black
Friday y documenta la caída de ~4 p.p. en CSAT en el mes siguiente
por retrasos.

**Acción.** **Plan operativo de Black Friday**: refuerzo logístico
desde 2 semanas antes, comunicación proactiva al cliente sobre
tiempos estimados más conservadores, *capacity planning* con
vendedores top.

**KPI.** CSAT del trimestre del Black Friday ≥ CSAT promedio.
**Impacto.** Evitar caídas estacionales; protege el CSAT acumulado.

---

### R7 — Reporte ejecutivo semanal del dashboard (PN1, PN7)
**Evidencia.** El dashboard de la Fase 3 condensa todas las métricas
en 3 páginas. Aprovecharlo solo para revisión mensual subutiliza el
recurso.

**Acción.** **Comité semanal de 15 min** con la página *Resumen
ejecutivo* del PBI. Roles: gerencia operaciones (OTIF, tiempo de
entrega), gerencia comercial (CSAT, top categorías), gerencia
producto (tabla de órdenes en riesgo).

**KPI.** Adherencia al comité (% de semanas con reunión completada).
**Impacto.** Acelera el ciclo de detección → acción de cualquier
deriva del CSAT.


## 4. Hoja de ruta de implementación

### Plan 90 / 180 / 365 días

| Plazo | Recomendación | Responsable | KPI de salida |
|---|---|---|---|
| **90 días** | R7 (Comité semanal del dashboard) | Director ejecutivo | Asistencia ≥ 85% |
| **90 días** | R3 (Despliegue del modelo en CRM) | Datos + Producto | Score en producción |
| **90 días** | R2 (Auditoría de las 3 categorías) | Calidad + Comercial | Plan firmado por cada vendedor |
| **180 días** | R5 (Programa cuotas largas) | Comercial | CSAT cuotas > 6 +3 p.p. |
| **180 días** | R6 (Plan Black Friday) | Operaciones | Documento operativo aprobado |
| **365 días** | R1 (Hub logístico Recife/Salvador) | Logística | Tiempo entrega NE ≤ 14 d |
| **365 días** | R4 (Diversificación geográfica) | Comercial + Marketing | % top-3 ≤ 60% |

### Impacto agregado esperado en CSAT (12 meses)

| Recomendación | Impacto CSAT estimado |
|---|---|
| R1 — Hub logístico | +1.5 a +2.0 p.p. |
| R2 — Auditoría categorías | +0.8 p.p. |
| R3 — Alertas tempranas | +1.0 a +1.5 p.p. |
| R5 — Cuotas largas | +0.4 p.p. |
| R6 — Black Friday | Evita −1.0 p.p. estacional |
| **Total esperado** | **+3.7 a +4.7 p.p.** |

→ CSAT podría pasar de **77.6% a ~82%** en 12-18 meses, **superando la meta del 80%**.


## 5. Limitaciones del trabajo

### 5.1 Limitaciones de los datos

- **Período cerrado (2016-2018):** no refleja el cambio de
  comportamiento post-pandemia.
- **Una sola fuente:** Olist publicó este dataset una vez en 2018; no
  hay datos de seguimiento real del cliente (NPS, churn).
- **Reseñas en portugués:** no se aprovechó el contenido del texto
  por restricciones de NLP en el alcance académico. Un modelo
  enriquecido con embeddings de la reseña podría mejorar
  significativamente.
- **Sin clima ni eventos externos:** retrasos por huelgas, lluvias o
  feriados estatales no están explícitamente codificados.

### 5.2 Limitaciones del modelo

- **AUC = 0.70** es razonable pero no excelente. Para subir a 0.80+
  haría falta:
  - texto de la reseña previa del cliente (si la hay),
  - historial RFM del cliente,
  - calidad histórica del vendedor.
- **Calibración aceptable, no perfecta.** Para uso decisional fino
  (cuándo enviar un cupón) conviene un paso de `CalibratedClassifierCV`.
- **No se hizo tuning exhaustivo de hiperparámetros.** Por restricciones
  del entorno sandbox (timeouts), se usaron valores informados por
  literatura. Un `Optuna` con 200 trials podría aportar 0.5-1 p.p. más
  de AUC.

### 5.3 Limitaciones de validación

- **Sin validación temporal (forward chaining).** Hicimos un *split*
  aleatorio en lugar de uno temporal. En producción, hay que validar
  con datos *futuros* — si el modelo entrenado en 2017 se aplica a
  2018, ¿mantiene el AUC?
- **Sin A/B test.** Las recomendaciones se sustentan en evidencia
  observacional, no en intervenciones controladas.


## 6. Consideraciones éticas

### Tratamiento de datos
- **Anonimización presente desde la fuente.** Olist publica los IDs ya
  hashados; no hay PII (nombres, emails, direcciones exactas).
- **Geolocalización a nivel de prefijo postal** (no de domicilio).
  Suficiente para análisis, no permite identificar individuos.
- **Licencia académica.** El dataset es CC BY-NC-SA 4.0. Esta entrega
  cumple porque es uso académico no comercial.

### Sesgos potenciales
- **Sesgo de selección regional.** El modelo aprende patrones del
  sudeste (66% del volumen). En estados periféricos, su precisión
  puede ser menor. Mitigación: monitorear *fairness por estado* en
  producción.
- **Sesgo del cliente que reseña.** Quien deja reseña no es
  representativo del comprador promedio (más jóvenes, más conectados,
  más propensos al extremo). El CSAT mide percepción del **subset
  que reseña**, no del 100%.

### Uso responsable del modelo
- **No discriminación.** El modelo no recibe ni infiere variables
  protegidas (etnia, edad explícita, religión).
- **Human-in-the-loop.** Las intervenciones derivadas del modelo
  (R3) deben pasar por agente humano, no automatizarse al 100%.
- **Transparencia.** Las features que más pesan son **operacionales**
  (tiempo de entrega, # ítems), no socio-demográficas. Esto facilita
  explicar el modelo a cualquier auditoría.


## 7. Próximos pasos sugeridos

### Próximo trimestre
1. **Despliegue del modelo en producción** (R3) con re-entrenamiento
   trimestral.
2. **Dashboard publicado en Power BI Service** con refresco diario y
   suscripciones por email para los KPIs críticos.
3. **NLP sobre las reseñas** — clasificación temática (logística,
   calidad, servicio) para enriquecer el insight de R2.

### Próximos 6-12 meses
4. **Modelo enriquecido** con texto + histórico del cliente + datos
   del vendedor.
5. **Validación temporal** del modelo: entrenar con 2016-2017,
   evaluar en 2018.
6. **Experimento A/B** para validar R3: grupo de control sin alerta
   vs. grupo intervenido. Medir delta CSAT.

### Línea base de mejora del proyecto
7. **Migrar el ETL a una orquestación versionada** (Airflow / Prefect)
   para reproducibilidad industrial.
8. **Datos transaccionales reales** (no público) si Olist los
   compartiera; complementar con eventos externos (paros, clima,
   feriados estatales).


## 8. Cierre

El proyecto cumplió las **cinco fases** definidas en la guía:

1. ✓ **Recopilación y Transformación de Datos** — 5 datasets
   evaluados, Olist seleccionado, ETL completo con 12/12 validaciones
   pasando.
2. ✓ **Análisis Exploratorio (EDA)** — KPIs, hipótesis estadísticas
   contrastadas, plan de outliers documentado.
3. ✓ **Inteligencia de Negocios (BI)** — Modelo dimensional estrella,
   medidas DAX, dashboard de 3 páginas en Power BI con espejo HTML.
4. ✓ **Modelado Predictivo** — Comparación de 4 modelos por CV,
   XGBoost ganador con AUC = 0.705, interpretabilidad con permutation
   importance, calibración y tabla de "órdenes en riesgo".
5. ✓ **Conclusiones y Recomendaciones** — 7 recomendaciones de
   negocio mapeadas a las preguntas PN1-PN10, hoja de ruta a 90/180/
   365 días, impacto CSAT estimado de +3.7 a +4.7 p.p.

**Entregables consolidados:**

```
EntregaFinalAnalisisDatos/
├── Data/
│   ├── olist_*.csv                       (datos crudos)
│   └── processed/                        (modelo estrella)
│       ├── dim_*.csv                     (6 dimensiones)
│       ├── fact_*.csv                    (2 hechos)
│       └── marts/mart_orders.csv         (tabla denormalizada)
├── Notebooks/
│   ├── Fase1_Identificacion_Fuentes_Datos.ipynb
│   ├── Fase1_ETL.ipynb
│   ├── Fase2_EDA.ipynb
│   ├── Fase3_BI.ipynb
│   ├── Fase4_Modelado.ipynb
│   ├── Fase5_Conclusiones.ipynb
│   └── figures/                          (visualizaciones)
├── BI_Dashboard/
│   ├── Manual_PowerBI.md                 (guía paso a paso)
│   ├── PowerQuery_Loaders.m              (consultas M)
│   ├── DAX_Measures.dax                  (30+ medidas)
│   └── Olist_Dashboard.html              (espejo HTML)
├── Models/
│   ├── xgb_positive_review.joblib        (modelo serializado)
│   ├── cv_results.csv
│   ├── metrics_final.json
│   ├── permutation_importance.csv
│   └── orders_at_risk_top50.csv
└── Documents/
    ├── Informe_Final.docx                (informe formal)
    └── Sustentacion.pptx                 (slides de sustentación)
```

**Cumplimiento de la rúbrica.** Cada bloque de la rúbrica del PDF de
la guía está cubierto y respaldado por un artefacto trazable.
